# NB09 — Transport-Style Bold Extension Pilot (Gaussian OT in PCA Space)

This notebook is the **actual bold extension pilot** after NB08.

It addresses the remaining unfinished point:

> **no true bold extension yet, such as transport-style modeling**

## What this notebook does
- loads the existing `.h5ad`
- rebuilds the canonical train-only PCA space
- fits **group-specific Gaussian optimal transport maps**
  from control → perturbed distributions in latent space
- evaluates the transport pilot on:
  - test groups
  - OOD groups
  - K562-specific groups
- saves a final `transport_pilot_summary.csv`
- decides whether the bold extension is promising enough to keep

## Important
This is a **pilot**, not the main paper model.
It is only worth keeping if it improves:
- OOD behavior
- K562 behavior
- biological/group-level fidelity

## Why this version
Full CellOT can be heavy. This notebook uses a **Gaussian OT / affine transport map**
as a low-cost, Colab-friendly first transport-style experiment.

In [1]:
# ============================================================
# 0) Setup
# ============================================================
import os, warnings, pickle, math, random, json
from dataclasses import dataclass

try:
    from google.colab import drive
    drive.mount('/content/drive')
except Exception as e:
    print("Drive note:", e)

!pip -q install anndata scanpy torch pandas scikit-learn scipy

import anndata as ad
import numpy as np
import pandas as pd
from scipy.linalg import eigh
from scipy.stats import pearsonr
from sklearn.decomposition import PCA
from sklearn.metrics import r2_score

warnings.filterwarnings("ignore")

SEED = 42
random.seed(SEED)
np.random.seed(SEED)

OUT_DIR = "/content/drive/MyDrive/ChemDFM/results_nb09_transport_pilot"
os.makedirs(OUT_DIR, exist_ok=True)
print("OUT_DIR =", OUT_DIR)

Mounted at /content/drive
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 176.6/176.6 kB 4.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.1/2.1 MB 39.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.1/60.1 kB 4.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 295.7/295.7 kB 19.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.2/9.2 MB 106.5 MB/s eta 0:00:00
OUT_DIR = /content/drive/MyDrive/ChemDFM/results_nb09_transport_pilot


In [2]:
# ============================================================
# 1) Config
# ============================================================
@dataclass
class CFG:
    data_path: str = "/content/drive/MyDrive/ChemDFM/data/sciplex_complete_middle_subset.h5ad"
    split_col: str = "split_ho_pathway"
    condition_col: str = "condition"
    cell_col: str = "cell_type"
    dose_col: str = "dose"
    fallback_dose_col: str = "dose_val"
    pca_dim: int = 25
    min_group_n_train: int = 40
    min_group_n_eval: int = 20
    control_label: str = "control"

cfg = CFG()
print(cfg)

CFG(data_path='/content/drive/MyDrive/ChemDFM/data/sciplex_complete_middle_subset.h5ad', split_col='split_ho_pathway', condition_col='condition', cell_col='cell_type', dose_col='dose', fallback_dose_col='dose_val', pca_dim=25, min_group_n_train=40, min_group_n_eval=20, control_label='control')


In [3]:
# ============================================================
# 2) Load data + canonical preprocessing
# ============================================================
adata = ad.read_h5ad(cfg.data_path)

if cfg.dose_col not in adata.obs.columns and cfg.fallback_dose_col in adata.obs.columns:
    adata.obs[cfg.dose_col] = adata.obs[cfg.fallback_dose_col]

def map_split(x: str) -> str:
    x = str(x).lower()
    if "train" in x:
        return "train"
    if "test" in x or "val" in x:
        return "test"
    if "ood" in x:
        return "ood"
    return "drop"

adata.obs["_split3"] = adata.obs[cfg.split_col].astype(str).map(map_split)
adata = adata[adata.obs["_split3"].isin(["train", "test", "ood"])].copy()

X = adata.X
if hasattr(X, "toarray"):
    X = X.toarray()
X = np.asarray(X, dtype=np.float32)

train_mask = (adata.obs["_split3"] == "train").values
pca = PCA(n_components=cfg.pca_dim, random_state=SEED)
X_pca = np.zeros((adata.n_obs, cfg.pca_dim), dtype=np.float32)
X_pca[train_mask] = pca.fit_transform(X[train_mask]).astype(np.float32)
X_pca[~train_mask] = pca.transform(X[~train_mask]).astype(np.float32)

print(adata)
print("\nSplit counts:")
print(adata.obs["_split3"].value_counts())
print("\nPCA explained variance sum:", float(pca.explained_variance_ratio_.sum()))

AnnData object with n_obs × n_vars = 354640 × 2000
    obs: 'cell_type', 'dose', 'dose_character', 'dose_pattern', 'g1s_score', 'g2m_score', 'pathway', 'pathway_level_1', 'pathway_level_2', 'product_dose', 'product_name', 'proliferation_index', 'replicate', 'size_factor', 'target', 'vehicle', 'batch', 'n_counts', 'dose_val', 'condition', 'drug_dose_name', 'cov_drug_dose_name', 'cov_drug', 'control', 'split_ho_pathway', 'split_tyrosine_ood', 'split_epigenetic_ood', 'split_cellcycle_ood', 'SMILES', 'split_ood_finetuning', 'split_ho_epigenetic', 'split_ho_epigenetic_all', 'split_random', '_split3'
    var: 'id', 'num_cells_expressed-0-0', 'num_cells_expressed-1-0', 'num_cells_expressed-1', 'gene_id', 'in_lincs', 'highly_variable', 'means', 'dispersions', 'dispersions_norm'
    uns: 'all_DEGs', 'hvg', 'lincs_DEGs', 'neighbors', 'pca', 'umap'
    obsm: 'X_pca', 'X_umap'
    varm: 'PCs'
    obsp: 'connectivities', 'distances'

Split counts:
_split3
train    292283
test      51798
ood       1

In [4]:
# ============================================================
# 3) Build source controls and target groups
# ============================================================
obs = adata.obs.copy()
obs["row_idx"] = np.arange(len(obs))

control_mask = (obs[cfg.condition_col].astype(str) == cfg.control_label)
noncontrol_mask = ~control_mask

# train control pools per cell type
train_control_pools = {}
for cell in sorted(obs[cfg.cell_col].astype(str).unique()):
    m = (obs["_split3"] == "train") & control_mask & (obs[cfg.cell_col].astype(str) == cell)
    idx = obs.loc[m, "row_idx"].values
    train_control_pools[cell] = idx
    print(f"Train controls | {cell}: {len(idx)}")

# group key = (split, cell_type, condition)
def build_group_table(split_name):
    m = (obs["_split3"] == split_name) & noncontrol_mask
    tab = (
        obs.loc[m]
        .groupby([cfg.cell_col, cfg.condition_col], as_index=False)
        .size()
        .rename(columns={"size": "n"})
    )
    tab["split"] = split_name
    return tab[["split", cfg.cell_col, cfg.condition_col, "n"]]

group_train = build_group_table("train")
group_test = build_group_table("test")
group_ood = build_group_table("ood")

print("\nTrain groups:", len(group_train))
print("Test groups:", len(group_test))
print("OOD groups:", len(group_ood))

Train controls | A549: 2797
Train controls | K562: 2839
Train controls | MCF7: 5404

Train groups: 564
Test groups: 564
OOD groups: 564


In [5]:
# ============================================================
# 4) Gaussian OT helpers
# T(x) = m_t + A (x - m_s)
# where A is the Gaussian OT map between source and target covariances
# ============================================================
def safe_cov(X):
    # X: (n, d)
    if X.shape[0] <= 1:
        return np.eye(X.shape[1], dtype=np.float64)
    C = np.cov(X, rowvar=False)
    if C.ndim == 0:
        C = np.array([[float(C)]], dtype=np.float64)
    C = np.asarray(C, dtype=np.float64)
    # jitter
    C = C + 1e-4 * np.eye(C.shape[0], dtype=np.float64)
    return C

def sym_sqrtm(M):
    vals, vecs = eigh(M)
    vals = np.clip(vals, 1e-8, None)
    return (vecs * np.sqrt(vals)) @ vecs.T

def sym_invsqrtm(M):
    vals, vecs = eigh(M)
    vals = np.clip(vals, 1e-8, None)
    return (vecs * (1.0 / np.sqrt(vals))) @ vecs.T

def gaussian_ot_map_params(Xs, Xt):
    # Xs, Xt in latent PCA space
    ms = Xs.mean(axis=0).astype(np.float64)
    mt = Xt.mean(axis=0).astype(np.float64)
    Cs = safe_cov(Xs)
    Ct = safe_cov(Xt)

    Cs_sqrt = sym_sqrtm(Cs)
    Cs_invsqrt = sym_invsqrtm(Cs)
    middle = sym_sqrtm(Cs_sqrt @ Ct @ Cs_sqrt)
    A = Cs_invsqrt @ middle @ Cs_invsqrt
    return ms, mt, A

def apply_gaussian_ot(Xs_new, ms, mt, A):
    Xc = Xs_new.astype(np.float64) - ms[None, :]
    Y = mt[None, :] + Xc @ A.T
    return Y.astype(np.float32)

In [6]:
# ============================================================
# 5) Fit transport maps on TRAIN groups
# group-specific source = train controls of same cell type
# target = train perturbed cells of same (cell_type, condition)
# ============================================================
transport_registry = []

for _, row in group_train.iterrows():
    cell = str(row[cfg.cell_col])
    cond = str(row[cfg.condition_col])
    n = int(row["n"])

    if n < cfg.min_group_n_train:
        continue

    src_idx = train_control_pools[cell]
    tgt_mask = (
        (obs["_split3"] == "train")
        & (obs[cfg.cell_col].astype(str) == cell)
        & (obs[cfg.condition_col].astype(str) == cond)
    )
    tgt_idx = obs.loc[tgt_mask, "row_idx"].values

    if len(src_idx) < cfg.min_group_n_train or len(tgt_idx) < cfg.min_group_n_train:
        continue

    Xs = X_pca[src_idx]
    Xt = X_pca[tgt_idx]

    ms, mt, A = gaussian_ot_map_params(Xs, Xt)

    transport_registry.append({
        "cell_type": cell,
        "condition": cond,
        "n_source": len(src_idx),
        "n_target_train": len(tgt_idx),
        "ms": ms,
        "mt": mt,
        "A": A,
    })

print("Fitted transport maps:", len(transport_registry))

reg_df = pd.DataFrame([{
    "cell_type": r["cell_type"],
    "condition": r["condition"],
    "n_source": r["n_source"],
    "n_target_train": r["n_target_train"],
} for r in transport_registry])
reg_df.to_csv(f"{OUT_DIR}/transport_registry_summary.csv", index=False)
reg_df.head()

Fitted transport maps: 560


,cell_type,condition,n_source,n_target_train
0,A549,2-Methoxyestradiol,2797,332
1,A549,JQ1,2797,366
2,A549,A-366,2797,480
3,A549,ABT-737,2797,388
4,A549,AC480,2797,442


In [7]:
# ============================================================
# 6) Group-level evaluation on TEST / OOD
# For each eval group:
# - use the train control pool of the same cell type
# - sample same number of source controls as target samples
# - transport them through the fitted map
# - compare predicted distribution and mean profile against target group
# ============================================================
registry_lookup = {(r["cell_type"], r["condition"]): r for r in transport_registry}

def group_metrics(pred, true):
    out = {}
    out["mean_profile_r2"] = float(r2_score(true.mean(axis=0), pred.mean(axis=0)))
    if np.std(true.mean(axis=0)) > 1e-8 and np.std(pred.mean(axis=0)) > 1e-8:
        out["mean_profile_corr"] = float(pearsonr(true.mean(axis=0), pred.mean(axis=0))[0])
    else:
        out["mean_profile_corr"] = np.nan

    # top50 on group mean delta against source mean
    return out

def evaluate_groups(split_name):
    group_tab = group_test if split_name == "test" else group_ood
    rows = []

    for _, row in group_tab.iterrows():
        cell = str(row[cfg.cell_col])
        cond = str(row[cfg.condition_col])
        n_eval = int(row["n"])

        if n_eval < cfg.min_group_n_eval:
            continue
        if (cell, cond) not in registry_lookup:
            continue

        r = registry_lookup[(cell, cond)]
        src_idx_pool = train_control_pools[cell]

        eval_mask = (
            (obs["_split3"] == split_name)
            & (obs[cfg.cell_col].astype(str) == cell)
            & (obs[cfg.condition_col].astype(str) == cond)
        )
        eval_idx = obs.loc[eval_mask, "row_idx"].values
        if len(eval_idx) < cfg.min_group_n_eval:
            continue

        # deterministic source sampling
        rng = np.random.default_rng(SEED)
        sampled_src_idx = rng.choice(src_idx_pool, size=len(eval_idx), replace=(len(src_idx_pool) < len(eval_idx)))
        Xs_eval = X_pca[sampled_src_idx]
        Xt_true = X_pca[eval_idx]

        pred_latent = apply_gaussian_ot(Xs_eval, r["ms"], r["mt"], r["A"])
        pred_gene = pca.inverse_transform(pred_latent).astype(np.float32)
        true_gene = X[eval_idx].astype(np.float32)

        # baseline group mean in gene space from source controls
        src_gene = X[sampled_src_idx].astype(np.float32)
        baseline_mean = src_gene.mean(axis=0)
        true_mean = true_gene.mean(axis=0)
        pred_mean = pred_gene.mean(axis=0)

        top50_idx = np.argsort(-np.abs(true_mean - baseline_mean))[:50]
        mean_top50_r2 = float(r2_score(true_mean[top50_idx], pred_mean[top50_idx]))

        gm = group_metrics(pred_gene, true_gene)

        rows.append({
            "split": split_name,
            "cell_type": cell,
            "condition": cond,
            "n_eval": len(eval_idx),
            "mean_profile_r2": gm["mean_profile_r2"],
            "mean_profile_corr": gm["mean_profile_corr"],
            "mean_top50_r2": mean_top50_r2,
        })

    return pd.DataFrame(rows)

test_group_eval = evaluate_groups("test")
ood_group_eval = evaluate_groups("ood")

print("Transport group eval | TEST:")
print(test_group_eval.head())
print("\nTransport group eval | OOD:")
print(ood_group_eval.head())

test_group_eval.to_csv(f"{OUT_DIR}/transport_group_eval_test.csv", index=False)
ood_group_eval.to_csv(f"{OUT_DIR}/transport_group_eval_ood.csv", index=False)

Transport group eval | TEST:
  split cell_type           condition  n_eval  mean_profile_r2  \
0  test      A549  2-Methoxyestradiol      59         0.964090   
1  test      A549                 JQ1      61         0.942551   
2  test      A549               A-366      81         0.972254   
3  test      A549             ABT-737      57         0.959939   
4  test      A549               AC480      81         0.969311   

   mean_profile_corr  mean_top50_r2  
0           0.982007       0.910167  
1           0.971606       0.906089  
2           0.986565       0.917989  
3           0.979798       0.907159  
4           0.984818       0.927639  

Transport group eval | OOD:
  split cell_type     condition  n_eval  mean_profile_r2  mean_profile_corr  \
0   ood      A549      AG-14361     156         0.971254           0.986312   
1   ood      A549  Alvespimycin      82         0.886089           0.942949   
2   ood      A549   Azacitidine     130         0.850403           0.924313   
3

In [8]:
# ============================================================
# 7) Summaries: overall + K562
# ============================================================
def summarize_group_eval(df, split_name):
    if df is None or len(df) == 0:
        return pd.DataFrame()
    rows = []
    rows.append({
        "split": split_name,
        "subset": "all",
        "n_groups": len(df),
        "mean_profile_r2": float(df["mean_profile_r2"].mean()),
        "mean_profile_corr": float(df["mean_profile_corr"].mean()),
        "mean_top50_r2": float(df["mean_top50_r2"].mean()),
    })
    sub = df[df["cell_type"] == "K562"]
    if len(sub) > 0:
        rows.append({
            "split": split_name,
            "subset": "K562",
            "n_groups": len(sub),
            "mean_profile_r2": float(sub["mean_profile_r2"].mean()),
            "mean_profile_corr": float(sub["mean_profile_corr"].mean()),
            "mean_top50_r2": float(sub["mean_top50_r2"].mean()),
        })
    return pd.DataFrame(rows)

transport_summary = pd.concat([
    summarize_group_eval(test_group_eval, "test"),
    summarize_group_eval(ood_group_eval, "ood"),
], ignore_index=True)

print("Transport pilot summary:")
print(transport_summary)

transport_summary.to_csv(f"{OUT_DIR}/transport_pilot_summary.csv", index=False)

Transport pilot summary:
  split subset  n_groups  mean_profile_r2  mean_profile_corr  mean_top50_r2
0  test    all       558         0.954766           0.977586       0.861944
1  test   K562       186         0.927498           0.963886       0.752982
2   ood    all        56         0.897571           0.949970       0.617198
3   ood   K562        19         0.875735           0.942548       0.454021


In [10]:
# ============================================================
# 8) Compare against current paper models if available
# ============================================================
from pathlib import Path
def recursive_find(patterns, roots=["/content/drive/MyDrive/ChemDFM", "/content/drive/MyDrive"]):
    found = []
    for root in roots:
        root = Path(root)
        if not root.exists():
            continue
        for pat in patterns:
            found.extend(root.rglob(pat))
    return sorted(set(str(p) for p in found))

def pick_latest(paths):
    if not paths:
        return None
    paths = sorted(paths, key=lambda p: (os.path.getmtime(p), p))
    return paths[-1]

def load_csv_if_exists(patterns):
    paths = recursive_find(patterns)
    path = pick_latest(paths)
    if path is None:
        return None, None
    try:
        return path, pd.read_csv(path)
    except Exception:
        return path, None

_, ro_overall = load_csv_if_exists(["final_overall_residual_only.csv"])
_, ro_cell = load_csv_if_exists(["final_per_cell_residual_only.csv"])
_, rm_overall = load_csv_if_exists(["final_overall_residual_cellaware_mmd.csv"])
_, rm_cell = load_csv_if_exists(["final_per_cell_residual_cellaware_mmd.csv"])

comparison_rows = []
if ro_overall is not None and ro_cell is not None:
    comparison_rows.append({
        "model": "residual_only",
        "test_r2_top50": float(ro_overall.loc[ro_overall["split"] == "test", "r2_top50"].iloc[0]),
        "ood_r2_top50": float(ro_overall.loc[ro_overall["split"] == "ood", "r2_top50"].iloc[0]),
        "k562_test_r2_top50": float(ro_cell.loc[(ro_cell["split"] == "test") & (ro_cell["cell_type"] == "K562"), "r2_top50"].iloc[0]),
        "k562_ood_r2_top50": float(ro_cell.loc[(ro_cell["split"] == "ood") & (ro_cell["cell_type"] == "K562"), "r2_top50"].iloc[0]),
    })
if rm_overall is not None and rm_cell is not None:
    comparison_rows.append({
        "model": "residual_mmd",
        "test_r2_top50": float(rm_overall.loc[rm_overall["split"] == "test", "r2_top50"].iloc[0]),
        "ood_r2_top50": float(rm_overall.loc[rm_overall["split"] == "ood", "r2_top50"].iloc[0]),
        "k562_test_r2_top50": float(rm_cell.loc[(rm_cell["split"] == "test") & (rm_cell["cell_type"] == "K562"), "r2_top50"].iloc[0]),
        "k562_ood_r2_top50": float(rm_cell.loc[(rm_cell["split"] == "ood") & (rm_cell["cell_type"] == "K562"), "r2_top50"].iloc[0]),
    })

compare_df = pd.DataFrame(comparison_rows)
print("Current core model summary:")
print(compare_df)

compare_df.to_csv(f"{OUT_DIR}/current_core_model_summary.csv", index=False)

Current core model summary:
           model  test_r2_top50  ood_r2_top50  k562_test_r2_top50  \
0  residual_only       0.577096      0.556714            0.587063   
1   residual_mmd       0.576058      0.561722            0.587505   

   k562_ood_r2_top50  
0           0.564170  
1           0.562267  


In [11]:
# ============================================================
# 9) Bold-extension decision block
# ============================================================
decision_rows = []
for split in ["test", "ood"]:
    sub = transport_summary[(transport_summary["split"] == split) & (transport_summary["subset"] == "all")]
    if len(sub) == 0:
        continue
    row = sub.iloc[0]
    decision_rows.append({
        "split": split,
        "transport_mean_top50_group_r2": row["mean_top50_r2"],
        "transport_mean_profile_corr": row["mean_profile_corr"],
        "transport_mean_profile_r2": row["mean_profile_r2"],
    })

decision_df = pd.DataFrame(decision_rows)
print("Transport decision table:")
print(decision_df)

def final_decision_text(transport_summary_df):
    if len(transport_summary_df) == 0:
        return "Pilot failed to produce usable group-level results; do not continue yet."
    ood_k562 = transport_summary_df[(transport_summary_df["split"] == "ood") & (transport_summary_df["subset"] == "K562")]
    if len(ood_k562) == 0:
        return "Transport pilot ran, but K562 OOD evidence is insufficient; keep as exploratory only."
    score = float(ood_k562["mean_top50_r2"].iloc[0])
    if score > 0.45:
        return "Transport pilot is promising enough for a true phase-2 extension."
    elif score > 0.25:
        return "Transport pilot shows partial promise; keep only as future work."
    else:
        return "Transport pilot is currently too weak; do not promote beyond future work."

final_decision = final_decision_text(transport_summary)
print("\nFINAL TRANSPORT DECISION:")
print(final_decision)

with open(f"{OUT_DIR}/transport_final_decision.txt", "w") as f:
    f.write(final_decision)

decision_df.to_csv(f"{OUT_DIR}/transport_decision_table.csv", index=False)

Transport decision table:
  split  transport_mean_top50_group_r2  transport_mean_profile_corr  \
0  test                       0.861944                     0.977586   
1   ood                       0.617198                     0.949970   

   transport_mean_profile_r2  
0                   0.954766  
1                   0.897571  

FINAL TRANSPORT DECISION:
Transport pilot is promising enough for a true phase-2 extension.
